# 🗺️ CrashLens — OSM Road Cache Generator (All 50 States + DC)

Downloads the entire USA road network one state at a time → parquet → **gzip** → R2.

| Feature | Detail |
|---------|--------|
| **States** | 50 states + DC = 51 total |
| **Gzipped** | Parquet compressed ~40-60% before upload |
| **Auto-skip** | Checks R2 first — resumes if Colab disconnects |
| **Memory safe** | Frees graph after each state upload |
| **List-safe** | Converts OSM list columns to strings (fixes ArrowTypeError) |
| **Live dashboard** | Progress bar, ETA, recent completions |
| **Est. time** | ~4-5 hours for all 51 (fits in free Colab session) |

**Run Cells 1→4 in order. Cell 4 is fully automatic.**

⚠️ **Clear R2 credentials from Cell 2 when done.**

## Cell 1 — Install Dependencies (~60s)

In [ ]:
!pip install -q osmnx geopandas pandas pyarrow scipy boto3

import osmnx as ox
import geopandas as gpd
import pandas as pd
import boto3
import os, time, json, gc, gzip, shutil, math
from pathlib import Path
from IPython.display import display, HTML, clear_output

print(f'osmnx {ox.__version__}')
print(f'geopandas {gpd.__version__}')
print('✅ All dependencies installed')

## Cell 2 — R2 Credentials

Same 3 values as your GitHub Secrets (`R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`).

Find them at: GitHub repo → Settings → Secrets and variables → Actions

Or Cloudflare dashboard → R2 → Manage API Tokens.

In [ ]:
# ════════════════════════════════════════════════════════════
#  PASTE YOUR R2 CREDENTIALS HERE
# ════════════════════════════════════════════════════════════

R2_ENDPOINT      = ''   # e.g. 'https://0e03851de64c74f3dffc4646....r2.cloudflarestorage.com'
R2_ACCESS_KEY_ID = ''   # R2 access key ID
R2_SECRET_KEY    = ''   # R2 secret access key
R2_BUCKET        = 'crash-lens-data'

# ════════════════════════════════════════════════════════════

assert R2_ENDPOINT,      '❌ Paste R2_ENDPOINT above'
assert R2_ACCESS_KEY_ID, '❌ Paste R2_ACCESS_KEY_ID above'
assert R2_SECRET_KEY,    '❌ Paste R2_SECRET_KEY above'

s3 = boto3.client('s3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_KEY,
    region_name='auto',
)

try:
    s3.list_objects_v2(Bucket=R2_BUCKET, Prefix='delaware/', MaxKeys=1)
    print(f'✅ R2 connected — bucket: {R2_BUCKET}')
except Exception as e:
    print(f'❌ R2 connection failed: {e}')

## Cell 3 — State Registry (50 states + DC)

All 51 entries. To run a subset, uncomment one of the filter lines at the bottom.

In [ ]:
ALL_STATES = [
    {"name": "Alabama", "abbreviation": "al", "r2_prefix": "alabama"},
    {"name": "Alaska", "abbreviation": "ak", "r2_prefix": "alaska"},
    {"name": "Arizona", "abbreviation": "az", "r2_prefix": "arizona"},
    {"name": "Arkansas", "abbreviation": "ar", "r2_prefix": "arkansas"},
    {"name": "California", "abbreviation": "ca", "r2_prefix": "california"},
    {"name": "Colorado", "abbreviation": "co", "r2_prefix": "colorado"},
    {"name": "Connecticut", "abbreviation": "ct", "r2_prefix": "connecticut"},
    {"name": "Delaware", "abbreviation": "de", "r2_prefix": "delaware"},
    {"name": "District of Columbia", "abbreviation": "dc", "r2_prefix": "district_of_columbia"},
    {"name": "Florida", "abbreviation": "fl", "r2_prefix": "florida"},
    {"name": "Georgia", "abbreviation": "ga", "r2_prefix": "georgia"},
    {"name": "Hawaii", "abbreviation": "hi", "r2_prefix": "hawaii"},
    {"name": "Idaho", "abbreviation": "id", "r2_prefix": "idaho"},
    {"name": "Illinois", "abbreviation": "il", "r2_prefix": "illinois"},
    {"name": "Indiana", "abbreviation": "in", "r2_prefix": "indiana"},
    {"name": "Iowa", "abbreviation": "ia", "r2_prefix": "iowa"},
    {"name": "Kansas", "abbreviation": "ks", "r2_prefix": "kansas"},
    {"name": "Kentucky", "abbreviation": "ky", "r2_prefix": "kentucky"},
    {"name": "Louisiana", "abbreviation": "la", "r2_prefix": "louisiana"},
    {"name": "Maine", "abbreviation": "me", "r2_prefix": "maine"},
    {"name": "Maryland", "abbreviation": "md", "r2_prefix": "maryland"},
    {"name": "Massachusetts", "abbreviation": "ma", "r2_prefix": "massachusetts"},
    {"name": "Michigan", "abbreviation": "mi", "r2_prefix": "michigan"},
    {"name": "Minnesota", "abbreviation": "mn", "r2_prefix": "minnesota"},
    {"name": "Mississippi", "abbreviation": "ms", "r2_prefix": "mississippi"},
    {"name": "Missouri", "abbreviation": "mo", "r2_prefix": "missouri"},
    {"name": "Montana", "abbreviation": "mt", "r2_prefix": "montana"},
    {"name": "Nebraska", "abbreviation": "ne", "r2_prefix": "nebraska"},
    {"name": "Nevada", "abbreviation": "nv", "r2_prefix": "nevada"},
    {"name": "New Hampshire", "abbreviation": "nh", "r2_prefix": "new_hampshire"},
    {"name": "New Jersey", "abbreviation": "nj", "r2_prefix": "new_jersey"},
    {"name": "New Mexico", "abbreviation": "nm", "r2_prefix": "new_mexico"},
    {"name": "New York", "abbreviation": "ny", "r2_prefix": "new_york"},
    {"name": "North Carolina", "abbreviation": "nc", "r2_prefix": "north_carolina"},
    {"name": "North Dakota", "abbreviation": "nd", "r2_prefix": "north_dakota"},
    {"name": "Ohio", "abbreviation": "oh", "r2_prefix": "ohio"},
    {"name": "Oklahoma", "abbreviation": "ok", "r2_prefix": "oklahoma"},
    {"name": "Oregon", "abbreviation": "or", "r2_prefix": "oregon"},
    {"name": "Pennsylvania", "abbreviation": "pa", "r2_prefix": "pennsylvania"},
    {"name": "Rhode Island", "abbreviation": "ri", "r2_prefix": "rhode_island"},
    {"name": "South Carolina", "abbreviation": "sc", "r2_prefix": "south_carolina"},
    {"name": "South Dakota", "abbreviation": "sd", "r2_prefix": "south_dakota"},
    {"name": "Tennessee", "abbreviation": "tn", "r2_prefix": "tennessee"},
    {"name": "Texas", "abbreviation": "tx", "r2_prefix": "texas"},
    {"name": "Utah", "abbreviation": "ut", "r2_prefix": "utah"},
    {"name": "Vermont", "abbreviation": "vt", "r2_prefix": "vermont"},
    {"name": "Virginia", "abbreviation": "va", "r2_prefix": "virginia"},
    {"name": "Washington", "abbreviation": "wa", "r2_prefix": "washington"},
    {"name": "West Virginia", "abbreviation": "wv", "r2_prefix": "west_virginia"},
    {"name": "Wisconsin", "abbreviation": "wi", "r2_prefix": "wisconsin"},
    {"name": "Wyoming", "abbreviation": "wy", "r2_prefix": "wyoming"},
]

print(f'{len(ALL_STATES)} states registered')

# ── FILTER OPTIONS (uncomment ONE line to limit) ──
# STATES_TO_RUN = [s for s in ALL_STATES if s['abbreviation'] == 'de']                    # Just Delaware
# STATES_TO_RUN = [s for s in ALL_STATES if s['abbreviation'] in ('de','va','co','md')]   # Specific states
# STATES_TO_RUN = ALL_STATES[:10]                                                         # First 10 only
STATES_TO_RUN = ALL_STATES                                                                 # All 51

print(f'Will process: {len(STATES_TO_RUN)} states')

## Cell 4 — Run Pipeline (fully automatic, gzipped)

Processes each state sequentially:
1. Check R2 for existing cache → skip if found
2. Download road network from OpenStreetMap
3. Fix list columns (prevents ArrowTypeError)
4. Save as parquet → gzip
5. Upload `.parquet.gz` to R2
6. Delete local files to free Colab disk
7. Move to next state

**Safe to re-run** — skips states already in R2.

**If Colab disconnects**, just re-run Cell 4 — it picks up where it left off.

In [ ]:
# ════════════════════════════════════════════════════════════
#  MAIN PIPELINE — processes all states automatically
# ════════════════════════════════════════════════════════════

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)
KEEP_COLS = ['geometry', 'name', 'ref', 'highway', 'maxspeed', 'lanes', 'surface', 'oneway', 'length']

results = {'completed': [], 'skipped': [], 'failed': []}
t_start = time.time()


# ── Helpers ──

def r2_exists(prefix, abbr):
    """Check if cache already uploaded to R2."""
    try:
        s3.head_object(Bucket=R2_BUCKET, Key=f'{prefix}/cache/{abbr}_roads.parquet.gz')
        return True
    except:
        return False


def r2_upload(local_path, r2_key):
    """Upload file to R2 with 3 retries."""
    for attempt in range(3):
        try:
            s3.upload_file(str(local_path), R2_BUCKET, r2_key)
            return True
        except Exception as e:
            if attempt == 2:
                print(f'    ❌ Upload failed: {e}')
                return False
            time.sleep(2 ** (attempt + 1))
    return False


def gzip_file(src_path, dst_path):
    """Gzip a file and return (raw_mb, gz_mb)."""
    with open(src_path, 'rb') as f_in, gzip.open(dst_path, 'wb', compresslevel=6) as f_out:
        shutil.copyfileobj(f_in, f_out)
    raw_mb = os.path.getsize(src_path) / 1048576
    gz_mb = os.path.getsize(dst_path) / 1048576
    os.remove(src_path)
    return raw_mb, gz_mb


def show_progress(idx, state_name, status_msg):
    """Live dashboard with progress bar."""
    clear_output(wait=True)
    total = len(STATES_TO_RUN)
    d = len(results['completed'])
    sk = len(results['skipped'])
    fl = len(results['failed'])
    done = d + sk + fl
    pct = done / total * 100 if total else 0
    elapsed = time.time() - t_start
    eta = f'{(elapsed / max(done, 1)) * (total - done) / 60:.0f} min' if done else '...'

    bar_len = 40
    filled = int(bar_len * pct / 100)
    bar = '█' * filled + '░' * (bar_len - filled)

    total_gz = sum(s.get('gz_mb', 0) for s in results['completed'])
    total_saved = sum(s.get('saved', 0) for s in results['completed'])

    html = f'''<div style="font-family:monospace;padding:20px;background:#1e1e2e;color:#cdd6f4;border-radius:12px;max-width:700px;">
    <h3 style="color:#89b4fa;margin:0 0 12px;">🗺️ CrashLens OSM Cache Generator</h3>
    <div style="font-size:20px;letter-spacing:1px;">{bar} {pct:.0f}%</div>
    <table style="color:#cdd6f4;font-size:14px;margin:12px 0;border-spacing:4px 6px;">
      <tr><td style="padding-right:20px;color:#a6adc8;">📍 Current:</td><td><b>{state_name}</b> — {status_msg}</td></tr>
      <tr><td style="color:#a6adc8;">✅ Downloaded:</td><td><b>{d}</b> of {total}</td></tr>
      <tr><td style="color:#a6adc8;">⏭️ Already in R2:</td><td>{sk}</td></tr>
      <tr><td style="color:#a6adc8;">❌ Failed:</td><td>{fl}</td></tr>
      <tr><td style="color:#a6adc8;">⏱️ Elapsed:</td><td>{elapsed/60:.1f} min</td></tr>
      <tr><td style="color:#a6adc8;">⏳ ETA:</td><td>~{eta}</td></tr>
      <tr><td style="color:#a6adc8;">💾 R2 uploaded:</td><td>{total_gz:.0f} MB</td></tr>
      <tr><td style="color:#a6adc8;">📦 Gzip saved:</td><td>{total_saved:.0f} MB</td></tr>
    </table>'''

    if results['completed']:
        last5 = results['completed'][-5:]
        html += '<div style="font-size:11px;color:#a6adc8;margin-top:4px;">Recent: '
        html += ' → '.join(f"{s['abbreviation'].upper()} ({s['gz_mb']:.0f}MB, {s['sec']:.0f}s)" for s in last5)
        html += '</div>'

    if results['failed']:
        html += '<div style="color:#f38ba8;font-size:11px;margin-top:6px;">'
        html += 'Failed: ' + ', '.join(f"{s['name']}" for s in results['failed'])
        html += '</div>'

    html += '</div>'
    display(HTML(html))


def process_state(state_info, idx):
    """Download, cache, gzip, upload one state."""
    name = state_info['name']
    abbr = state_info['abbreviation']
    prefix = state_info['r2_prefix']

    # ── Check R2 ──
    show_progress(idx, name, 'Checking R2...')
    if r2_exists(prefix, abbr):
        results['skipped'].append({'name': name, 'abbreviation': abbr})
        return

    # ── Download from OSM ──
    show_progress(idx, name, 'Downloading from OpenStreetMap...')
    t0 = time.time()

    # Special place queries for ambiguous state names
    if name == 'District of Columbia':
        place = 'Washington, DC, United States'
    elif name == 'Georgia':
        place = 'State of Georgia, United States'
    elif name == 'Washington':
        place = 'State of Washington, United States'
    else:
        place = f'{name}, United States'

    try:
        G = ox.graph_from_place(place, network_type='drive', simplify=True)
    except Exception:
        try:
            show_progress(idx, name, f'Retrying with "State of {name}"...')
            G = ox.graph_from_place(f'State of {name}, United States', network_type='drive', simplify=True)
        except Exception as e:
            results['failed'].append({'name': name, 'abbreviation': abbr, 'error': str(e)[:80]})
            return

    show_progress(idx, name, f'{G.number_of_nodes():,} nodes → saving + gzipping...')

    # ── Roads ──
    edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
    keep = [c for c in KEEP_COLS if c in edges.columns]
    slim = edges[keep].copy()

    # Fix: OSM list columns → semicolon-joined strings (prevents ArrowTypeError)
    for col in slim.columns:
        if col == 'geometry':
            continue
        if slim[col].apply(lambda v: isinstance(v, (list, tuple))).any():
            slim[col] = slim[col].apply(
                lambda v: '; '.join(str(x) for x in v) if isinstance(v, (list, tuple)) else v
            )

    roads_pq = str(CACHE_DIR / f'{abbr}_roads.parquet')
    roads_gz = roads_pq + '.gz'
    slim.to_parquet(roads_pq, index=True)
    raw_r, gz_r = gzip_file(roads_pq, roads_gz)

    # ── Intersections ──
    nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)
    deg = dict(G.degree())
    nodes['degree'] = nodes.index.map(deg)
    ints = nodes[nodes['degree'] >= 3][['geometry', 'degree']]
    int_pq = str(CACHE_DIR / f'{abbr}_intersections.parquet')
    int_gz = int_pq + '.gz'
    ints.to_parquet(int_pq, index=True)
    raw_i, gz_i = gzip_file(int_pq, int_gz)

    sec = time.time() - t0
    saved_mb = (raw_r + raw_i) - (gz_r + gz_i)

    # ── Upload to R2 ──
    show_progress(idx, name, f'Uploading {gz_r:.0f} + {gz_i:.0f} MB to R2...')
    ok1 = r2_upload(roads_gz, f'{prefix}/cache/{abbr}_roads.parquet.gz')
    ok2 = r2_upload(int_gz, f'{prefix}/cache/{abbr}_intersections.parquet.gz')

    # ── Cleanup ──
    Path(roads_gz).unlink(missing_ok=True)
    Path(int_gz).unlink(missing_ok=True)
    del G, edges, nodes, ints, slim
    gc.collect()

    if ok1 and ok2:
        results['completed'].append({
            'name': name, 'abbreviation': abbr,
            'gz_mb': gz_r + gz_i, 'raw_mb': raw_r + raw_i,
            'saved': saved_mb, 'sec': sec,
        })
    else:
        results['failed'].append({'name': name, 'abbreviation': abbr, 'error': 'R2 upload failed'})


# ═══════════════════════════════════════════════════════════
#  RUN ALL STATES
# ═══════════════════════════════════════════════════════════

for i, state in enumerate(STATES_TO_RUN):
    try:
        process_state(state, i)
    except Exception as e:
        results['failed'].append({
            'name': state['name'],
            'abbreviation': state['abbreviation'],
            'error': str(e)[:80],
        })
    # Small delay between states — be polite to OSM servers
    if i < len(STATES_TO_RUN) - 1:
        time.sleep(2)

# ── Final summary ──
show_progress(len(STATES_TO_RUN) - 1, 'DONE', 'All states processed ✅')
elapsed_total = time.time() - t_start

print(f'\n{"═" * 60}')
print(f'  COMPLETE in {elapsed_total / 60:.1f} minutes')
print(f'  Downloaded & uploaded: {len(results["completed"])}')
print(f'  Skipped (in R2):      {len(results["skipped"])}')
print(f'  Failed:               {len(results["failed"])}')
if results['completed']:
    total_gz = sum(s['gz_mb'] for s in results['completed'])
    total_raw = sum(s['raw_mb'] for s in results['completed'])
    total_saved = sum(s['saved'] for s in results['completed'])
    print(f'  R2 storage used:      {total_gz:.0f} MB ({total_gz/1024:.1f} GB)')
    print(f'  Raw parquet was:      {total_raw:.0f} MB ({total_raw/1024:.1f} GB)')
    print(f'  Gzip savings:         {total_saved:.0f} MB ({total_saved/1024:.1f} GB)')
if results['failed']:
    print(f'  Failed states: {", ".join(s["name"] for s in results["failed"])}')
print(f'{"═" * 60}')

## Cell 5 — Verify All R2 Cache Files (optional)

Lists every state's cache in R2 with file sizes.

In [ ]:
print('OSM cache files in R2:\n')
total_mb = 0
cached_count = 0
missing_states = []

for state in ALL_STATES:
    prefix = state['r2_prefix']
    abbr = state['abbreviation']
    try:
        resp = s3.list_objects_v2(Bucket=R2_BUCKET, Prefix=f'{prefix}/cache/{abbr}_')
        files = resp.get('Contents', [])
        if files:
            mb = sum(f['Size'] for f in files) / 1048576
            total_mb += mb
            cached_count += 1
            file_info = ', '.join(f"{os.path.basename(f['Key'])} ({f['Size']/1048576:.1f}MB)" for f in files)
            print(f'  ✅ {state["name"]:22s} ({abbr})  {file_info}')
        else:
            missing_states.append(state['name'])
            print(f'  ⬜ {state["name"]:22s} ({abbr})  — no cache')
    except Exception as e:
        missing_states.append(state['name'])
        print(f'  ❌ {state["name"]:22s} ({abbr})  — error: {e}')

print(f'\n{"═" * 60}')
print(f'  Cached: {cached_count} / {len(ALL_STATES)} states')
print(f'  Total R2 size: {total_mb:.0f} MB ({total_mb / 1024:.1f} GB)')
if missing_states:
    print(f'  Missing: {", ".join(missing_states)}')
print(f'{"═" * 60}')

## Cell 6 — Retry Failed States (optional)

If any states failed (OSM timeout, network issue), re-run just those.

In [ ]:
if results.get('failed'):
    retry_abbrs = {f['abbreviation'] for f in results['failed']}
    retry_states = [s for s in ALL_STATES if s['abbreviation'] in retry_abbrs]
    print(f'Retrying {len(retry_states)} failed states: {", ".join(s["name"] for s in retry_states)}')
    results['failed'] = []  # reset

    for i, state in enumerate(retry_states):
        try:
            process_state(state, i)
        except Exception as e:
            results['failed'].append({
                'name': state['name'],
                'abbreviation': state['abbreviation'],
                'error': str(e)[:80],
            })
        time.sleep(5)  # longer delay for retries

    print(f'\nRetry complete:')
    print(f'  OK:    {len(results["completed"])}')
    print(f'  Still failing: {len(results["failed"])}')
    if results['failed']:
        for f in results['failed']:
            print(f'    {f["name"]}: {f["error"]}')
else:
    print('No failed states to retry — all good! ✅')